# Team A — Llama-2 7B: Calibration Sweep + Knowledge Distillation

**Runtime:** GPU → A100 (Runtime → Change runtime type → A100)  
**Estimated time:** 4-6 hours for full sweep + KD  
**GPU memory needed:** ~20 GB (for KD phase)

## What this notebook does
1. Installs dependencies and clones the repo
2. Runs FP16 baseline calibration on ARC-Challenge
3. Runs PTQ sweep: GPTQ INT4, AWQ INT4, NF4
4. Runs knowledge distillation (FP16 teacher → INT4 student)
5. Plots calibration recovery analysis
6. Saves all results to Google Drive

## 0. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies (takes ~3 minutes)
!pip install -q transformers datasets accelerate tokenizers
!pip install -q auto-gptq autoawq bitsandbytes peft
!pip install -q wandb matplotlib scipy scikit-learn
print("Installation complete.")

In [ ]:
# Clone project repo
!git clone https://github.com/YOUR-ORG/uncertainty-aware-inference.git
%cd uncertainty-aware-inference
import sys
sys.path.insert(0, '.')

In [ ]:
# HuggingFace login (needed for Llama-2)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Mount Google Drive (for saving results)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/uncertainty-inference/llama2_7b'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

In [ ]:
# W&B login (optional)
# import wandb
# wandb.login()

## 1. Load Evaluation Dataset

In [ ]:
from src.calibration.datasets import load_dataset_mcq

# Load ARC-Challenge (primary calibration dataset)
N_SAMPLES = 300  # use 300 for speed; 500 for final results
samples = load_dataset_mcq('arc_challenge', n_samples=N_SAMPLES)

print(f"Loaded {len(samples)} samples")
print("\nExample sample:")
import json
print(json.dumps(samples[0], indent=2))

## 2. FP16 Baseline

In [ ]:
from src.quantization.loaders import load_model, free_model
from src.calibration.metrics import evaluate_calibration
from src.calibration.plots import plot_dashboard
import matplotlib.pyplot as plt

MODEL_ID   = 'meta-llama/Llama-2-7b-hf'
MODEL_NAME = 'Llama-2-7B'
DEVICE     = 'cuda'

print("Loading FP16 model (~14 GB)...")
model, tokenizer = load_model(MODEL_ID, 'FP16')
print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
fp16_result = evaluate_calibration(
    model=model, tokenizer=tokenizer,
    samples=samples,
    model_name=MODEL_NAME, precision='FP16',
    dataset='arc_challenge', device=DEVICE,
)
fp16_result.save(f'{RESULTS_DIR}/{MODEL_NAME}_FP16_arc_challenge_calibration.json')
print(fp16_result.summary())

In [ ]:
fig = plot_dashboard(fp16_result)
fig.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_FP16_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Free FP16 model before loading quantized configs
free_model(model)
print(f"GPU memory freed. Now: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. PTQ Sweep

In [ ]:
from src.calibration.plots import plot_metrics_comparison

PRECISIONS = ['GPTQ_INT4', 'AWQ_INT4', 'NF4']
all_results = [fp16_result]  # start with FP16 baseline

for precision in PRECISIONS:
    print(f"\n{'='*50}")
    print(f"  {precision}")
    print('='*50)

    try:
        model, tokenizer = load_model(MODEL_ID, precision)
        print(f"  GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

        result = evaluate_calibration(
            model=model, tokenizer=tokenizer,
            samples=samples,
            model_name=MODEL_NAME, precision=precision,
            dataset='arc_challenge', device=DEVICE,
        )
        result.save(f'{RESULTS_DIR}/{MODEL_NAME}_{precision}_arc_challenge_calibration.json')
        all_results.append(result)

        # Dashboard plot
        fig = plot_dashboard(result)
        fig.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_{precision}_dashboard.png',
                    dpi=150, bbox_inches='tight')
        plt.close(fig)

    except Exception as e:
        print(f"  ERROR: {e}")
    finally:
        free_model(model)

print("\nPTQ sweep complete.")

In [ ]:
# Cross-config comparison
fig = plot_metrics_comparison(all_results)
fig.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_ptq_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== PTQ Sweep Summary ===")
print(f"{'Precision':12s}  {'ECE':>8s}  {'MCE':>8s}  {'Brier':>8s}  {'Accuracy':>8s}")
print('-' * 55)
for r in all_results:
    print(f"{r.precision:12s}  {r.ece:8.4f}  {r.mce:8.4f}  {r.brier:8.4f}  {r.accuracy:8.4f}")

## 4. Temperature Scaling Baseline

In [ ]:
# Temperature scaling: can we recover calibration without KD?
# This is the null hypothesis — if TS alone recovers calibration, KD adds nothing.
import numpy as np
from src.calibration.temperature_scaling import fit_temperature_scaling

# We need raw logits (choice log-scores) for this.
# For demonstration: simulate from result data.
# In production: save logits during evaluate_calibration and load here.

print("Note: Temperature scaling requires raw logits saved during evaluation.")
print("Modify evaluate_calibration() to return logits alongside results if needed.")
print("See src/calibration/temperature_scaling.py for the full implementation.")

## 5. Knowledge Distillation

In [ ]:
# ⚠️  This cell loads BOTH FP16 teacher (~14GB) and INT4 student (~4GB).
# Requires ~20GB VRAM total — use A100.
from src.distillation.trainer import KDConfig, train_kd
from src.calibration.datasets import load_kd_corpus
import torch

print(f"GPU memory before loading: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print("Loading FP16 teacher...")
teacher, tokenizer = load_model(MODEL_ID, 'FP16')
print(f"  After teacher: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print("Loading GPTQ INT4 student...")
student, _ = load_model(MODEL_ID, 'GPTQ_INT4')
print(f"  After student: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Load KD corpus
texts = load_kd_corpus(n_samples=2000)
print(f"KD corpus: {len(texts)} text samples")

# KD config (fast for Colab — reduce epochs if running out of time)
kd_cfg = KDConfig(
    temperature  = 4.0,
    alpha        = 0.7,
    lr           = 2e-5,
    n_epochs     = 2,      # reduce to 1 if time is tight
    batch_size   = 4,
    max_length   = 128,    # shorter for speed
    eval_every   = 50,
)

from pathlib import Path
kd_output_dir = Path(f'{RESULTS_DIR}/kd')

print("\nStarting KD training...")
train_result = train_kd(
    teacher_model = teacher,
    student_model = student,
    texts         = texts,
    tokenizer     = tokenizer,
    config        = kd_cfg,
    output_dir    = kd_output_dir,
    device        = DEVICE,
)
print(f"\nKD complete. Final KD loss: {train_result.final_kd_loss:.4f}")

In [ ]:
# Free teacher — only need student for evaluation
free_model(teacher)
print(f"GPU after freeing teacher: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Evaluate post-KD calibration
post_kd_result = evaluate_calibration(
    model=student, tokenizer=tokenizer,
    samples=samples,
    model_name=MODEL_NAME, precision='GPTQ_INT4_KD',
    dataset='arc_challenge', device=DEVICE,
)
post_kd_result.save(f'{RESULTS_DIR}/kd/{MODEL_NAME}_GPTQ_INT4_KD_arc_challenge_calibration.json')
print(post_kd_result.summary())

## 6. KD Recovery Analysis

In [ ]:
from src.distillation.trainer import compute_kd_recovery
from src.calibration.metrics import CalibrationResult
import matplotlib.pyplot as plt

# Load pre-KD result
pre_kd_result = CalibrationResult.load(
    f'{RESULTS_DIR}/{MODEL_NAME}_GPTQ_INT4_arc_challenge_calibration.json'
)

metrics_cfg = {'ece': False, 'mce': False, 'brier': False, 'accuracy': True}
recovery = {}

print("=== KD Recovery Analysis ===")
print(f"{'Metric':12s}  {'FP16':>8s}  {'Pre-KD':>8s}  {'Post-KD':>8s}  {'Recovery':>10s}")
print('-' * 55)
for m, higher_better in metrics_cfg.items():
    r = compute_kd_recovery(
        fp16_val         = getattr(fp16_result,    m),
        pre_kd_val       = getattr(pre_kd_result,  m),
        post_kd_val      = getattr(post_kd_result, m),
        metric_name      = m,
        higher_is_better = higher_better,
    )
    recovery[m] = r
    print(f"{m:12s}  "
          f"{getattr(fp16_result,m):8.4f}  "
          f"{getattr(pre_kd_result,m):8.4f}  "
          f"{getattr(post_kd_result,m):8.4f}  "
          f"{r*100:+9.1f}%")

In [ ]:
# Recovery bar chart
metrics = list(recovery.keys())
vals_fp16    = [getattr(fp16_result,    m) for m in metrics]
vals_pre_kd  = [getattr(pre_kd_result,  m) for m in metrics]
vals_post_kd = [getattr(post_kd_result, m) for m in metrics]

fig, axes = plt.subplots(1, len(metrics), figsize=(16, 5))
for ax, m, v_fp16, v_pre, v_post in zip(
    axes, metrics, vals_fp16, vals_pre_kd, vals_post_kd
):
    bars = ax.bar(['FP16', 'Pre-KD\n(INT4)', 'Post-KD\n(INT4)'],
                  [v_fp16, v_pre, v_post],
                  color=['steelblue', 'coral', 'seagreen'])
    for bar, v in zip(bars, [v_fp16, v_pre, v_post]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(v_fp16, v_pre, v_post) * 0.01,
                f'{v:.4f}', ha='center', fontsize=9)
    ax.set_title(f'{m.upper()}\nRecovery: {recovery[m]*100:.1f}%')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(f'KD Calibration Recovery — {MODEL_NAME}', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/kd/kd_recovery.png', dpi=150, bbox_inches='tight')
plt.show()
print("Recovery plot saved.")

## 7. Summary and Save All Results

In [ ]:
import json, os

summary = {
    'model'    : MODEL_NAME,
    'dataset'  : 'arc_challenge',
    'n_samples': N_SAMPLES,
    'results'  : {
        r.precision: {'ece': r.ece, 'mce': r.mce, 'brier': r.brier, 'accuracy': r.accuracy}
        for r in all_results + [post_kd_result]
    },
    'kd_recovery': recovery,
}

with open(f'{RESULTS_DIR}/team_a_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("All results saved to Google Drive:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f'  {f}')